# Pure Dense Retrieval Pipeline

Runs the selected OCR corpus through article preparation, recursive chunking, BGE-M3 embedding, and ChromaDB indexing. This notebook does not use BM25, an inverted index, or FAISS.

In [10]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data-pipeline').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'data-pipeline'))

from dense_retrieval.config import PipelineConfig

config = PipelineConfig(project_root=PROJECT_ROOT)
config.create_output_dirs()
print('OCR directory:', config.ocr_dir)
print('Documents CSV:', config.documents_csv)
print('Output directory:', config.output_dir)

OCR directory: /Users/nak/CADT/projects/legal-rag/data/text-ocr
Documents CSV: /Users/nak/CADT/projects/legal-rag/data/documents.csv
Output directory: /Users/nak/CADT/projects/legal-rag/data/dense_retrieval


In [11]:
from dense_retrieval.prepare_articles import prepare_articles

articles = prepare_articles(config)
print('Prepared articles:', len(articles))
print('Saved:', config.article_path)

Prepared articles: 3301
Saved: /Users/nak/CADT/projects/legal-rag/data/dense_retrieval/articles.jsonl


In [12]:
import importlib
import dense_retrieval.chunking as chunking_module
importlib.reload(chunking_module)
build_chunks = chunking_module.build_chunks

chunks = build_chunks(articles, config)
print('Created chunks:', len(chunks))
print('Saved:', config.chunk_path)

Created chunks: 3505
Saved: /Users/nak/CADT/projects/legal-rag/data/dense_retrieval/chunks.jsonl


In [13]:
from dense_retrieval.embeddings import embed_chunks

vectors = embed_chunks(chunks, config)
print('Embedding shape:', vectors.shape)
print('Saved:', config.embedding_path)

W1104 23:27:59.201000 42014 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/1753 [00:00<?, ?it/s]

Embedding shape: (3505, 1024)
Saved: /Users/nak/CADT/projects/legal-rag/data/dense_retrieval/embeddings.npy


In [14]:
from dense_retrieval.chroma_index import build_index

index_info = build_index(chunks, vectors, config)
print(json.dumps(index_info, ensure_ascii=False, indent=2))

{
  "collection": "legal_dense_bge_m3_450",
  "path": "/Users/nak/CADT/projects/legal-rag/data/dense_retrieval/chroma_db",
  "stored_vectors": 3505,
  "dimension": 1024,
  "metric": "cosine"
}


In [15]:
print('Pipeline complete.')
print('Articles:', len(articles))
print('Chunks:', len(chunks))
print('Vectors:', vectors.shape)
print('Collection:', index_info['collection'])

Pipeline complete.
Articles: 3301
Chunks: 3505
Vectors: (3505, 1024)
Collection: legal_dense_bge_m3_450
